In [ ]:
import pandas as pd
import numpy as np
import yabplot as yab
from matplotlib import cm
from matplotlib.colors import ListedColormap
from pyvista import Plotter

import pyvista as pv
pv.global_theme.transparent_background = True

def label_for_yab(data: np.ndarray | list) -> dict:
    assert len(data) == 66, "Data must have 66 entries corresponding to atlas labels."

    labels = np.genfromtxt('/home/gchan/kg98_scratch/gchan/Atlases/Tian/tian_atlas/'
                           'yab_labels.csv', dtype=str)

    data = dict(zip(labels[:66], data[:66]))
    return data

def set_yab_scalebar(pl: Plotter, font_size: int = 10, show: bool = True,
                     export_path: str = None) -> None:
    sbar = next(iter(pl.scalar_bars.values()))          # get colorbar object
    sbar.GetLabelTextProperty().SetFontSize(font_size)  # set colorbar label font size
    if show:
        pl.show(jupyter_backend='static')
    if export_path:
        # export with pl.screenshot (raster) to preserve transparency
        # pl.save_graphic (vector) does not support it
        pl.screenshot(export_path, transparent_background=True)
    pl.close()

In [30]:
# Plot 3 HC and 3 SCZ subjects voxelwise data projected onto the surface.

from yabplot.data import get_surface_paths
from yabplot.mesh import project_vol2surf, load_vertexwise_mesh
from yabplot.plotting import plot_vertexwise

nii_paths = [
    "/home/gchan/kg98/trangc/VBM/data/Advan_inno/sub-10002/anat/s6mwp1sub-10002_T1w.nii",
    "/home/gchan/kg98/trangc/VBM/data/Advan_inno/sub-10003/anat/s6mwp1sub-10003_T1w.nii",
    "/home/gchan/kg98/trangc/VBM/data/Advan_inno/sub-10004/anat/s6mwp1sub-10004_T1w.nii",
    "/home/gchan/kg98/trangc/VBM/data/Advan_inno/sub-10014/anat/s6mwp1sub-10014_T1w.nii",
    "/home/gchan/kg98/trangc/VBM/data/Advan_inno/sub-10016/anat/s6mwp1sub-10016_T1w.nii",
    "/home/gchan/kg98/trangc/VBM/data/Advan_inno/sub-10020/anat/s6mwp1sub-10020_T1w.nii",
    ]

b_lh_path, b_rh_path = get_surface_paths('midthickness', 'bmesh')

for nii_path in nii_paths:
    # project 3d volume to 1d surface arrays
    lh_data, rh_data = project_vol2surf(nii_path, bmesh_type='midthickness')

    # make vertex-wise brain meshes with the injected data
    lh_mesh, rh_mesh = load_vertexwise_mesh(b_lh_path, b_rh_path, lh_data, rh_data)

    plot_vertexwise(
        lh_mesh, rh_mesh, cmap='coolwarm', figsize=(600, 600), views=['left_lateral'],
        vminmax=(0, 1), display_type='none',
        export_path=f"./results/fig1_{nii_path.split('/')[-3]}_vertexwise.png"
    )

In [42]:
# Plot the lme betas voxelwise data projected onto the surface.

nii_path = "../emp_atrophy_lme/voxelwise_lme/results/lme_betas_2.nii.gz"
lh_data, rh_data = project_vol2surf(nii_path, bmesh_type='midthickness')
lh_mesh, rh_mesh = load_vertexwise_mesh(b_lh_path, b_rh_path, lh_data, rh_data)
plot_vertexwise(
    lh_mesh, rh_mesh, cmap='coolwarm', figsize=(600, 600), views=['left_lateral'],
    vminmax=(-4, 4), display_type='none',
    export_path=f"./results/fig1_lme_betas_vertexwise.png"
)

In [32]:
# visualize parcellation on the surface
def make_qualitative_cmap(n_colors: int = 66) -> ListedColormap:
    base_maps = ["Pastel1", "Pastel2"]
    # base_maps = ["Set2", "Set3"]
    palette = []

    for name in base_maps:
        cmap = cm.get_cmap(name)
        if hasattr(cmap, "colors"):
            palette.extend(cmap.colors)
        else:
            palette.extend(cmap(np.linspace(0, 1, cmap.N)))

    if n_colors <= len(palette):
        colors = palette[:n_colors]
    else:
        repeats = int(np.ceil(n_colors / len(palette)))
        colors = (palette * repeats)[:n_colors]

    return ListedColormap(colors, name=f"qual_{n_colors}")

data = label_for_yab(np.arange(66, dtype=int))
qual_cmap = make_qualitative_cmap(66)
yab.plot_cortical(
    data=data, atlas="schaefer_100", views=['left_lateral'], figsize=(600, 600),
    cmap=qual_cmap,
    display_type='none', export_path="./results/fig1_schaefer100_parcellation.png"
)


/tmp/ipykernel_3837040/1970120478.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(name)


In [ ]:
from sklearn.preprocessing import StandardScaler

sim_atr = pd.read_csv("/fs04/scratch2/kg98/oldscratch/gchan/SIR_SCZ/vis_nii/data/rnaseh2c_rnf122_49.csv", header=None)
emp_atr = pd.read_csv("/fs04/scratch2/kg98/oldscratch/gchan/SIR_SCZ/vis_nii/data/lme_betas.csv", header=None)

scaler = StandardScaler()

for df in [sim_atr, emp_atr]:
    data = label_for_yab(scaler.fit_transform(df.to_numpy().flatten().reshape(-1, 1)).flatten())
    data_values = np.asarray(list(data.values()), dtype=float)

    tian_dir = "/home/gchan/kg98_scratch/gchan/Atlases/Tian/tian_atlas/surfaces"
    lim = round(max(abs(np.nanmin(data_values)), abs(np.nanmax(data_values))))

    pl = yab.plot_cortical(
        data=data, atlas="schaefer_100", display_type='object',
        views=['left_lateral', 'left_medial'], figsize=(1200, 600), cmap="coolwarm",
        vminmax=(-lim, lim),
    )
    set_yab_scalebar(
        pl, font_size=16, show = False,
        export_path=f"./results/fig1_{'sim' if df is sim_atr else 'emp'}_atr_cx.png"
    )

    pl = yab.plot_subcortical(
        data=data, custom_atlas_path=tian_dir, display_type='object', bmesh_alpha=0.2,
        figsize=(1200, 600), cmap="coolwarm", views=['left_lateral', 'left_medial'],
        vminmax=(-lim, lim),
    )
    set_yab_scalebar(
        pl, font_size=16, show = False,
        export_path=f"./results/fig1_{'sim' if df is sim_atr else 'emp'}_atr_subcx.png"
    )